In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    countDistinct,
    year,
    month,
    date_format,
    to_timestamp,
    coalesce,
    lit,
    desc
)

# Load Silver Delta tables
orders = spark.table("workspace.silver.orders")
order_items = spark.table("workspace.silver.order_items")
customers = spark.table("workspace.silver.customers")
products = spark.table("workspace.silver.products")
category_translation = spark.table(
    "workspace.silver.category_translation"
)

print("All Silver datasets loaded successfully!")

print("Orders:", orders.count())
print("Order items:", order_items.count())
print("Customers:", customers.count())
print("Products:", products.count())
print("Category translations:", category_translation.count())

All Silver datasets loaded successfully!
Orders: 99441
Order items: 112650
Customers: 99441
Products: 32951
Category translations: 71


In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    countDistinct,
    date_format,
    to_timestamp,
    coalesce,
    lit
)

# Keep only delivered orders
delivered_orders = (
    orders
    .filter(col("order_status") == "delivered")
    .withColumn(
        "purchase_timestamp",
        to_timestamp(col("order_purchase_timestamp"))
    )
    .filter(col("purchase_timestamp").isNotNull())
)

# Join orders with order items
sales = delivered_orders.join(
    order_items,
    on="order_id",
    how="inner"
)

# Calculate revenue: item price + freight
sales = sales.withColumn(
    "total_revenue",
    coalesce(col("price"), lit(0)) +
    coalesce(col("freight_value"), lit(0))
)

# Aggregate monthly sales
monthly_sales = (
    sales
    .withColumn(
        "sales_month",
        date_format(col("purchase_timestamp"), "yyyy-MM")
    )
    .groupBy("sales_month")
    .agg(
        sum("total_revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        count("order_item_id").alias("total_items_sold")
    )
    .orderBy("sales_month")
)

# Save Gold Delta table
(
    monthly_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.monthly_sales")
)

print("MONTHLY SALES GOLD TABLE CREATED!")

display(monthly_sales)

MONTHLY SALES GOLD TABLE CREATED!


sales_month,total_revenue,total_orders,total_items_sold
2016-09,143.46,1,3
2016-10,46490.66,265,313
2016-12,19.62,1,1
2017-01,127482.37000000005,750,913
2017-02,271239.31999999995,1653,1858
2017-03,414330.9499999994,2546,2897
2017-04,390812.3999999997,2303,2569
2017-05,566851.4000000001,3546,4004
2017-06,490050.36999999976,3135,3489
2017-07,566299.080000001,3872,4416


In [0]:
from pyspark.sql.functions import sum, count, desc, round

# Aggregate product performance
top_products = (
    sales
    .groupBy("product_id")
    .agg(
        round(sum("total_revenue"), 2).alias("total_revenue"),
        count("order_item_id").alias("total_items_sold"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy(desc("total_revenue"))
)

# Save as Gold Delta table
(
    top_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.top_products")
)

print("TOP PRODUCTS GOLD TABLE CREATED!")

display(top_products.limit(10))

TOP PRODUCTS GOLD TABLE CREATED!


product_id,total_revenue,total_items_sold,total_orders
bb50f2e236e5eea0100680137654686c,67258.03,194,186
d1c427060a0f73f6b889a5c7c61f2ac4,58957.31,332,313
6cdd53843498f92890544667809f1595,57933.73,153,148
99a4788cb24856965c36a24e339b6058,49907.5,477,456
3dd2a17168ec895c781a9191c1e95ad7,47876.06,272,253
d6160fb7873f184099d9bc95e30376af,47314.18,33,33
aca2eb7d00ea1a7b8ebd4e68314663af,44197.75,520,425
5f504b3a1c75b73d6151be81eb05bdc9,41725.81,63,63
25c38557cf793876c5abdd5931f922db,40311.95,38,38
53b36df67ebb7c41585e8d54d6772e08,39713.49,321,304


In [0]:
from pyspark.sql.functions import countDistinct, sum, round, desc

# Join sales with customer information
customer_sales = sales.join(
    customers.select("customer_id", "customer_unique_id"),
    on="customer_id",
    how="inner"
)

# Aggregate customer purchasing behaviour
customer_analytics = (
    customer_sales
    .groupBy("customer_unique_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("total_revenue"), 2).alias("total_spent")
    )
    .withColumn(
        "average_order_value",
        round(col("total_spent") / col("total_orders"), 2)
    )
    .orderBy(desc("total_spent"))
)

# Save Gold table
(
    customer_analytics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.customer_analytics")
)

print("CUSTOMER ANALYTICS GOLD TABLE CREATED!")

display(customer_analytics.limit(10))

CUSTOMER ANALYTICS GOLD TABLE CREATED!


customer_unique_id,total_orders,total_spent,average_order_value
0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.08
da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.82
763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.88
dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.31
459bef486812aa25204be022145caa62,1,6922.21,6922.21
ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.66
4007669dec559734d6f53e029e360987,1,6081.54,6081.54
eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.34
48e1ac109decbb87765a3eade6854098,1,4681.78,4681.78
c8460e4251689ba205045f3ea17884a1,4,4655.88,1163.97


In [0]:
from pyspark.sql.functions import col, sum, count, countDistinct, round, desc

# Join sales with product details
category_data = (
    sales
    .join(
        products.select("product_id", "product_category_name"),
        on="product_id",
        how="left"
    )
    .join(
        category_translation,
        on="product_category_name",
        how="left"
    )
)

# Aggregate sales by product category
category_sales = (
    category_data
    .groupBy("product_category_name_english")
    .agg(
        round(sum("total_revenue"), 2).alias("total_revenue"),
        count("order_item_id").alias("total_items_sold"),
        countDistinct("order_id").alias("total_orders")
    )
    .withColumnRenamed(
        "product_category_name_english",
        "product_category"
    )
    .orderBy(desc("total_revenue"))
)

# Save Gold table
(
    category_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.category_sales")
)

print("CATEGORY SALES GOLD TABLE CREATED!")

display(category_sales.limit(15))

CATEGORY SALES GOLD TABLE CREATED!


product_category,total_revenue,total_items_sold,total_orders
health_beauty,1412089.53,9465,8647
watches_gifts,1264333.12,5859,5495
bed_bath_table,1225209.26,10953,9272
sports_leisure,1118256.91,8431,7530
computers_accessories,1032723.77,7644,6530
furniture_decor,880329.92,8160,6307
housewares,758392.25,6795,5743
cool_stuff,691680.89,3718,3559
auto,669454.75,4140,3810
garden_tools,567145.68,4268,3448


In [0]:
gold_tables = [
    "monthly_sales",
    "top_products",
    "customer_analytics",
    "category_sales"
]

verification_results = []

for table_name in gold_tables:
    df = spark.table(f"workspace.gold.{table_name}")

    verification_results.append((
        table_name,
        df.count(),
        len(df.columns),
        "SUCCESS"
    ))

verification_df = spark.createDataFrame(
    verification_results,
    ["table_name", "row_count", "column_count", "status"]
)

display(verification_df)

print("ALL FOUR GOLD TABLES VERIFIED!")

table_name,row_count,column_count,status
monthly_sales,23,4,SUCCESS
top_products,32216,4,SUCCESS
customer_analytics,93358,4,SUCCESS
category_sales,72,4,SUCCESS


ALL FOUR GOLD TABLES VERIFIED!
